# 04.1 — Language model text analysis lab

Every section below runs the **same task twice**: once through a purpose-built
Foundry Tool, once through a language model. Each run prints its latency, and the
model runs print token counts, so the trade-off table in [README.md](README.md)
stops being abstract.

**Prerequisites**

- `.env` with `AZURE_LANGUAGE_ENDPOINT`, `AZURE_TRANSLATOR_REGION`,
  `AZURE_CONTENT_SAFETY_ENDPOINT`, `MODEL_MINI`
- Data-plane role **Cognitive Services User** on the Foundry resource
  (control-plane Contributor is *not* enough — see Troubleshooting in the README)
- `pip install -r requirements.txt`

**Cost:** well under $1. Language, Translator, and Content Safety are all
consumption-priced, and every model call is on `gpt-4o-mini`.

**Region note:** abstractive summarization and Text Analytics for Health are not
available in every region. `swedencentral` (the course default) and `eastus2` are safe. Sections that
need them degrade with a printed explanation rather than a traceback.

## 1. Setup

Three clients, one credential, no keys.

| Client | Package | Endpoint from `.env` |
|---|---|---|
| `TextAnalyticsClient` | `azure-ai-textanalytics` | `AZURE_LANGUAGE_ENDPOINT` |
| `ContentSafetyClient` | `azure-ai-contentsafety` | `AZURE_CONTENT_SAFETY_ENDPOINT` |
| `TextTranslationClient` | `azure-ai-translation-text` | global endpoint + `region` + `resource_id` |

Translator is the odd one out. Because you call a **global** endpoint, an Entra ID
token alone does not say which resource to bill — so you must pass `region` *and*
`resource_id` alongside the credential.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client

import json, time, textwrap

from azure.ai.textanalytics import TextAnalyticsClient
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from azure.ai.translation.text import TextTranslationClient
from azure.core.exceptions import HttpResponseError

SUB = cfg.require("AZURE_SUBSCRIPTION_ID")
RG = cfg.require("AZURE_RESOURCE_GROUP")
RESOURCE = cfg.require("AZURE_AI_FOUNDRY_RESOURCE")
RESOURCE_ID = (
    f"/subscriptions/{SUB}/resourceGroups/{RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{RESOURCE}"
)

MINI = cfg.require("MODEL_MINI")

language = TextAnalyticsClient(
    endpoint=cfg.require("AZURE_LANGUAGE_ENDPOINT", unit="04.1"),
    credential=credential(),
)

safety = ContentSafetyClient(
    endpoint=cfg.get("AZURE_CONTENT_SAFETY_ENDPOINT")
    or cfg.require("AZURE_LANGUAGE_ENDPOINT"),
    credential=credential(),
)

translator = TextTranslationClient(
    credential=credential(),
    region=cfg.require("AZURE_TRANSLATOR_REGION", unit="04.1"),
    resource_id=RESOURCE_ID,
)

print("language endpoint :", cfg["AZURE_LANGUAGE_ENDPOINT"])
print("translator region :", cfg["AZURE_TRANSLATOR_REGION"])
print("model             :", MINI)

### A timer and a token counter

Two helpers used everywhere below. `timed` wraps any call and reports wall-clock
milliseconds; `chat` is a one-shot completion that also returns usage, because in
this unit the token count *is* part of the lesson.

In [ ]:
from contextlib import contextmanager


@contextmanager
def timed(label):
    """Print wall-clock latency for a block. Latency is a first-class result here."""
    start = time.perf_counter()
    try:
        yield
    finally:
        ms = (time.perf_counter() - start) * 1000
        print(f"  [{label}: {ms:,.0f} ms]")


def chat(prompt, *, system=None, model=None, response_format=None, temperature=0):
    """One-shot completion that returns (text, usage_dict, latency_ms)."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": model or MINI, "messages": messages, "temperature": temperature}
    if response_format:
        kwargs["response_format"] = response_format

    start = time.perf_counter()
    resp = chat_client().chat.completions.create(**kwargs)
    ms = (time.perf_counter() - start) * 1000

    u = resp.usage
    usage = {"prompt": u.prompt_tokens, "completion": u.completion_tokens, "total": u.total_tokens}
    return resp.choices[0].message.content or "", usage, ms


def report(label, usage, ms):
    print(
        f"  [{label}: {ms:,.0f} ms | "
        f"{usage['prompt']} in + {usage['completion']} out = {usage['total']} tokens]"
    )

## 2. The corpus

Three documents, deliberately different in shape, reused by every section:

1. **`SUPPORT`** — a customer complaint. Sentiment, opinion mining, tone.
2. **`INCIDENT`** — an internal incident report with names, dates, and an account
   number. Entities, PII, compliance summarization.
3. **`CLINICAL`** — a clinical note. Text Analytics for Health.

Keep them short: Language bills per 1,000-character *text record*, so a
2,500-character document is three records, not one.

In [ ]:
SUPPORT = (
    "I have been a customer for six years and this is the first time I have felt "
    "ignored. The new battery lasts barely four hours, which is unacceptable for a "
    "device at this price. The screen is genuinely beautiful and the keyboard is the "
    "best I have used. But your support line disconnected me twice and nobody called "
    "back. I would like a replacement battery or a refund."
)

INCIDENT = (
    "On 14 March 2026 at 02:17 UTC, Contoso Financial Services detected unauthorised "
    "access to the payments API in the West Europe region. The on-call engineer, "
    "Priya Raman, reachable at priya.raman@contoso.com or +44 20 7946 0958, escalated "
    "to the security team within nine minutes. Account 4111-1111-1111-1111 was the "
    "only customer record touched. The root cause was a rotated managed identity "
    "whose role assignment had not propagated, which caused the API to fall back to a "
    "stale shared key that had not been revoked. Mitigation completed at 03:41 UTC. "
    "Under GDPR Article 33 the incident was reported to the supervisory authority "
    "within 72 hours."
)

CLINICAL = (
    "Patient is a 62-year-old male presenting with intermittent chest pain radiating "
    "to the left arm. History of type 2 diabetes, managed with metformin 500 mg twice "
    "daily. No history of myocardial infarction. Blood pressure 148/92. Prescribed "
    "atorvastatin 20 mg daily and referred for a stress echocardiogram."
)

DOCS = {"SUPPORT": SUPPORT, "INCIDENT": INCIDENT, "CLINICAL": CLINICAL}

for name, text in DOCS.items():
    records = -(-len(text) // 1000)
    print(f"{name:9} {len(text):4} chars -> {records} text record(s)")

## 3. Entities and key phrases — the Language service

Three synchronous calls. Note what the model cannot give you: **`offset` and
`length`**. Every entity comes back with the exact character span it occupies,
which is what makes redaction and highlighting possible.

In [ ]:
with timed("recognize_entities"):
    ner = language.recognize_entities([INCIDENT])[0]

print("Named entities")
for e in ner.entities:
    print(
        f"  {e.text:35} {e.category:14} {e.subcategory or '-':12} "
        f"conf={e.confidence_score:.2f}  offset={e.offset:3} len={e.length}"
    )

In [ ]:
with timed("extract_key_phrases"):
    kp = language.extract_key_phrases([INCIDENT])[0]

print("Key phrases (unranked noun phrases — NOT topics):")
print(textwrap.fill(", ".join(kp.key_phrases), 78,
                    initial_indent="  ", subsequent_indent="  "))

with timed("recognize_linked_entities"):
    linked = language.recognize_linked_entities([INCIDENT])[0]

print("\nLinked entities (disambiguated to a knowledge base):")
for e in linked.entities:
    print(f"  {e.name:30} {e.data_source:12} {e.url}")

> **Exam note.** `extract_key_phrases` returns noun phrases from **one document**,
> unranked, with no notion of a corpus. "What are the three recurring themes across
> these 500 tickets?" is not this API. That is an LLM job, or clustering over
> embeddings — which is what you built in unit 05.1.

## 4. The same extraction as guaranteed-shape JSON

Now the model path. The point is not that the model *can* extract entities — it is
**how hard the output shape is guaranteed**.

| `response_format` | Guarantee |
|---|---|
| omitted | Nothing. You get prose, maybe wrapped in a fenced code block |
| `{"type": "json_object"}` | Valid JSON. **No** schema conformance |
| `{"type": "json_schema", ..., "strict": True}` | Valid JSON **matching your schema** |

Run all three and watch the first one break.

In [ ]:
EXTRACT_PROMPT = (
    "Extract the incident facts from the report below.\n\n"
    f"REPORT:\n{INCIDENT}"
)

# --- (a) no constraint at all ------------------------------------------------
text, usage, ms = chat(EXTRACT_PROMPT + "\n\nReturn JSON.")
report("free text", usage, ms)
print(text[:300])
try:
    json.loads(text)
    print("\n  -> parsed (lucky)")
except json.JSONDecodeError as exc:
    print(f"\n  -> json.loads FAILED: {exc}")

In [ ]:
# --- (b) JSON mode: valid JSON, arbitrary shape -------------------------------
text, usage, ms = chat(
    EXTRACT_PROMPT,
    system="You output JSON only.",
    response_format={"type": "json_object"},
)
report("json_object", usage, ms)
parsed = json.loads(text)          # guaranteed to parse
print(json.dumps(parsed, indent=2)[:500])
print("\n  -> parses, but the keys are whatever the model felt like today")

In [ ]:
# --- (c) Structured Outputs: the decoder is constrained to your schema --------
INCIDENT_SCHEMA = {
    "type": "object",
    "properties": {
        "detected_at_utc": {"type": "string", "description": "ISO 8601"},
        "organisation": {"type": "string"},
        "affected_system": {"type": "string"},
        "azure_region": {"type": "string"},
        "root_cause": {"type": "string"},
        "severity": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
        "customer_records_affected": {"type": "integer"},
        "regulator_notified": {"type": "boolean"},
        "people": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "role": {"type": "string"},
                },
                "required": ["name", "role"],
                "additionalProperties": False,   # required on EVERY object
            },
        },
    },
    "required": [
        "detected_at_utc", "organisation", "affected_system", "azure_region",
        "root_cause", "severity", "customer_records_affected",
        "regulator_notified", "people",
    ],
    "additionalProperties": False,
}

text, usage, ms = chat(
    EXTRACT_PROMPT,
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "incident", "strict": True, "schema": INCIDENT_SCHEMA},
    },
)
report("json_schema strict", usage, ms)
incident = json.loads(text)
print(json.dumps(incident, indent=2))

**Two rules the API enforces, and that the exam tests:**

1. In `strict` mode, **every** object — including nested ones — needs
   `"additionalProperties": false`.
2. **Every** property must appear in `required`. There are no optional fields.
   An "optional" field is modelled as a union with null: `{"type": ["string", "null"]}`.

Break either one and the request is rejected at schema-validation time, before a
single token is generated. Prove it to yourself:

In [ ]:
BROKEN = {
    "type": "object",
    "properties": {"organisation": {"type": "string"}, "region": {"type": "string"}},
    "required": ["organisation"],          # 'region' missing from required
    "additionalProperties": False,
}

try:
    chat(
        EXTRACT_PROMPT,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "broken", "strict": True, "schema": BROKEN},
        },
    )
    print("no error — your API version may predate strict validation")
except Exception as exc:
    print("rejected before generation, as designed:\n ", str(exc)[:400])

### Which one would you pick?

`recognize_entities` gave you `offset` and `length`; the model gave you a typed
business object with `severity` and `regulator_notified` — categories no
pre-trained NER model ships with.

> **Exam note.** "The application must highlight every detected entity in the
> original document" → Language, because you need offsets. "The application must
> populate a typed incident record with a severity rating" → Structured Outputs,
> because the categories are yours. Real systems use both: Language for the spans,
> a model for the judgement.

## 5. Summarization — extractive, abstractive, and a model

All three summarize `INCIDENT`. The two Language calls are **long-running
operations**: `begin_*` returns a poller and you call `.result()`. Code that
expects an immediate return value is the wrong answer on the exam.

In [ ]:
with timed("begin_extract_summary (LRO)"):
    poller = language.begin_extract_summary([INCIDENT], max_sentence_count=3)
    extractive = list(poller.result())[0]

print("Extractive — verbatim sentences, ranked, guaranteed faithful:")
for s in sorted(extractive.sentences, key=lambda s: s.offset):
    print(f"  [rank {s.rank_score:.2f}] {s.text}")

In [ ]:
try:
    with timed("begin_abstract_summary (LRO)"):
        poller = language.begin_abstract_summary([INCIDENT])
        abstractive = list(poller.result())[0]

    print("Abstractive — newly generated sentences:")
    for s in abstractive.summaries:
        print(textwrap.fill(s.text, 78, initial_indent="  ", subsequent_indent="  "))
except HttpResponseError as exc:
    print("Abstractive summarization is unavailable in this region.")
    print("Deploy the lab in swedencentral or eastus2 to run it.")
    print(" ", str(exc)[:200])

In [ ]:
text, usage, ms = chat(
    "Summarize this incident report in three sentences for an executive audience."
    f"\n\n{INCIDENT}"
)
report("LLM summary", usage, ms)
print(textwrap.fill(text, 78, initial_indent="  ", subsequent_indent="  "))

| | Extractive | Abstractive | LLM |
|---|---|---|---|
| Words are the document's | **yes** | no | no |
| Controls sentence count | `max_sentence_count` | no | prompt (soft) |
| Fabrication risk | **zero** | low | low, unmeasured |
| Latency | seconds (LRO) | seconds (LRO) | ~1 s |
| Auditable | rank score per sentence | no | no |

> **Exam note.** "The summary must be traceable to the source for a regulator" →
> extractive. "Summarize a call-centre transcript into issue and resolution" →
> **conversational** summarization (`ConversationAnalysisClient`, a different
> package), not `begin_abstract_summary`.

## 6. Sentiment, opinions, and tone — three different questions

`SUPPORT` is the interesting document: it is *negative overall* while containing
genuinely positive statements about the screen and keyboard. A single label loses
that. Opinion mining does not.

In [ ]:
with timed("analyze_sentiment + opinion mining"):
    doc = language.analyze_sentiment([SUPPORT], show_opinion_mining=True)[0]

s = doc.confidence_scores
print(f"document sentiment : {doc.sentiment}")
print(f"  positive={s.positive:.2f}  neutral={s.neutral:.2f}  negative={s.negative:.2f}\n")

print("Per sentence:")
for sent in doc.sentences:
    print(f"  [{sent.sentiment:8}] {sent.text.strip()[:64]}")

print("\nOpinion mining — target -> assessment:")
for sent in doc.sentences:
    for op in sent.mined_opinions:
        assessments = ", ".join(f"{a.text} ({a.sentiment})" for a in op.assessments)
        print(f"  {op.target.text:12} [{op.target.sentiment:8}] <- {assessments}")

In [ ]:
# Tone is NOT a Language feature. This is the model's job.
TONE_SCHEMA = {
    "type": "object",
    "properties": {
        "tone": {
            "type": "string",
            "enum": ["calm", "frustrated", "hostile", "sarcastic", "resigned", "escalating"],
        },
        "churn_risk": {"type": "string", "enum": ["low", "medium", "high"]},
        "requested_remedy": {"type": "string"},
        "evidence": {"type": "string", "description": "A short quote supporting the tone label"},
    },
    "required": ["tone", "churn_risk", "requested_remedy", "evidence"],
    "additionalProperties": False,
}

text, usage, ms = chat(
    f"Assess the tone of this customer message.\n\n{SUPPORT}",
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "tone", "strict": True, "schema": TONE_SCHEMA},
    },
)
report("LLM tone", usage, ms)
print(json.dumps(json.loads(text), indent=2))

Sentiment said `negative` with per-sentence detail and a confidence score, in about
a fifth of a second, for a fraction of a cent. The model added `churn_risk` and
`escalating` — categories that do not exist in the Language service at any price.

> **Exam note.** "Identify *what* the customer is unhappy about" → opinion mining
> (target/assessment pairs). "Classify the emotional register" → a model. "Detect
> harmful content" → Content Safety, next section. Three different services, and
> the exam will offer all three as distractors.

## 7. PII and PHI — where the model must not be used

`recognize_pii_entities` hands you `redacted_text` for free, plus the exact span of
every detection. A model asked to "remove the personal data" produces a *paraphrase*
that may silently drop a sentence or invent one, and gives you nothing to audit.

This is the clearest "use the Foundry Tool" case in the whole exam.

In [ ]:
with timed("recognize_pii_entities"):
    pii = language.recognize_pii_entities([INCIDENT])[0]

print("Detections:")
for e in pii.entities:
    print(
        f"  {e.text:28} {e.category:22} conf={e.confidence_score:.2f} "
        f"offset={e.offset:3} len={e.length}"
    )

print("\nRedacted text (produced by the service, deterministic):")
print(textwrap.fill(pii.redacted_text, 78,
                    initial_indent="  ", subsequent_indent="  "))

In [ ]:
# Narrow the categories — this is how you build a policy-specific redactor.
with timed("recognize_pii_entities (filtered)"):
    financial = language.recognize_pii_entities(
        [INCIDENT], categories_filter=["CreditCardNumber", "Email", "PhoneNumber"]
    )[0]

print("Only financial and contact identifiers:")
for e in financial.entities:
    print(f"  {e.text:28} {e.category}")
print()
print(textwrap.fill(financial.redacted_text, 78,
                    initial_indent="  ", subsequent_indent="  "))

In [ ]:
try:
    with timed("begin_analyze_healthcare_entities (LRO)"):
        poller = language.begin_analyze_healthcare_entities([CLINICAL])
        health = [d for d in poller.result() if not d.is_error][0]

    print("Clinical entities:")
    for e in health.entities:
        links = ", ".join(f"{d.name}:{d.entity_id}" for d in (e.data_sources or [])[:2])
        print(f"  {e.text:28} {e.category:22} norm={e.normalized_text or '-':24} {links}")

    print("\nRelations:")
    for rel in health.entity_relations:
        roles = " / ".join(f"{r.name}={r.entity.text}" for r in rel.roles)
        print(f"  {rel.relation_type:24} {roles}")
except HttpResponseError as exc:
    print("Text Analytics for Health is unavailable in this region.")
    print(" ", str(exc)[:200])

Look at what health analysis returned that no prompt reliably produces: **UMLS
concept IDs**. `metformin` normalised and linked to a coding system is an
interoperable fact. A model's `"medication": "metformin"` is a string.

> **Exam note.** "Redact patient identifiers before storage" → `recognize_pii_entities`
> with `domain_filter="phi"`. "Link medications to a clinical vocabulary" →
> `begin_analyze_healthcare_entities`. The first is synchronous, the second is an
> LRO — that asymmetry is a favourite exam detail.

## 8. Content Safety — a different question again

Four categories (**Hate, SelfHarm, Sexual, Violence**), severity `0 / 2 / 4 / 6`.

The furious complaint from section 6 is `negative` sentiment and severity `0`
everywhere. That contrast is the whole lesson: *sentiment is about the author's
feelings; safety is about harm.*

Note also that this is **not** the content filter on your model deployment. That
filter runs automatically on prompts and completions you generate. `analyze_text`
is you calling the classifier on text you did **not** generate — user uploads,
scraped content, third-party feeds.

In [ ]:
SAMPLES = {
    "angry complaint": SUPPORT,
    "incident report": INCIDENT,
    "violent phrasing": "I am going to hunt down whoever wrote this firmware and make them pay.",
}

for label, sample in SAMPLES.items():
    try:
        with timed(f"analyze_text: {label}"):
            result = safety.analyze_text(AnalyzeTextOptions(text=sample))
        flags = " ".join(f"{c.category}={c.severity}" for c in result.categories_analysis)
        worst = max(c.severity for c in result.categories_analysis)
        verdict = "BLOCK" if worst >= 4 else ("REVIEW" if worst >= 2 else "allow")
        print(f"  {label:18} {flags}   -> {verdict}")
    except HttpResponseError as exc:
        print(f"  {label:18} error: {str(exc)[:120]}")

The severity thresholds are **yours**. The service returns numbers; the policy that
turns `4` into a block and `2` into a human review queue is application logic. An
exam question that says "content must be reviewed by a moderator rather than
rejected" is describing a threshold decision, not a service setting.

## 9. Translation — Translator vs a model

Same sentence, both paths, and the numbers side by side.

In [ ]:
SOURCE = (
    "Your subscription renews automatically unless cancelled at least "
    "thirty days before the anniversary date."
)
TARGETS = ["de", "ja", "fr"]

with timed("Translator (3 languages, one call)"):
    results = translator.translate(body=[SOURCE], to_language=TARGETS)

item = results[0]
detected = getattr(item, "detected_language", None)
if detected:
    print(f"detected source: {detected.language} (score {detected.score:.2f})\n")
for t in item.translations:
    print(f"  {t.to:3} {t.text}")

In [ ]:
total_tokens = 0
start = time.perf_counter()
for lang in TARGETS:
    text, usage, ms = chat(
        f"Translate into {lang}. Output only the translation.\n\n{SOURCE}"
    )
    total_tokens += usage["total"]
    print(f"  {lang:3} {text.strip()}")
llm_ms = (time.perf_counter() - start) * 1000
print(f"\n  [LLM: {llm_ms:,.0f} ms for 3 calls | {total_tokens} tokens]")

Translator did three languages in **one** request, at roughly a tenth of the latency
and a fraction of the cost. Now the case where it loses:

In [ ]:
NUANCED = "Heads up — the deploy went sideways, so we're pushing the demo to Monday."

with timed("Translator"):
    literal = translator.translate(body=[NUANCED], to_language=["ja"])[0].translations[0].text
print("Translator :", literal)

text, usage, ms = chat(
    "Translate into Japanese using formal business register. "
    "Preserve the meaning, not the idiom. Output only the translation.\n\n" + NUANCED
)
report("LLM", usage, ms)
print("LLM        :", text.strip())

### Terminology control is not a prompt

The exam's favourite translation question: *"translations must use the company's
approved product terminology."* The answer is **Custom Translator** — or, for a
handful of terms, a **dynamic dictionary**, which needs no training at all:

In [ ]:
# Dynamic dictionary: wrap the term so Translator passes it through verbatim.
TERM = (
    'Please open the <mstrans:dictionary translation="Contoso Sentinel">'
    "Contoso Sentinel</mstrans:dictionary> console and restart the collector."
)

out = translator.translate(body=[TERM], to_language=["de"])[0].translations[0].text
print("with dictionary :", out)

plain = translator.translate(
    body=["Please open the Contoso Sentinel console and restart the collector."],
    to_language=["de"],
)[0].translations[0].text
print("without         :", plain)

| Requirement | Answer |
|---|---|
| 80,000 SKUs, uniform quality, cheap | Translator |
| Approved names for a few regulated terms | Dynamic dictionary |
| Domain-wide terminology and style, thousands of terms | **Custom Translator** (needs parallel data — roughly 10,000 aligned sentences for a full model) |
| Formal register, idiom, pun, marketing voice | LLM |
| Volume *and* nuance | Translator first, LLM post-edit on low-confidence output |

> **Exam note.** "Prompt the model to use the glossary" is always a distractor when
> the requirement says *must*. Prompts are advisory; a dictionary or a custom model
> is enforcement.

## 10. Domain customization — a compliance summarizer you can defend

The escalation ladder from the README, applied. Start with the naive prompt, then
add constraints until the output is something a compliance officer would accept.

**Step 1 — a plain prompt.** Fluent, and unusable.

In [ ]:
text, usage, ms = chat(f"Summarize this for our compliance team.\n\n{INCIDENT}")
report("naive", usage, ms)
print(textwrap.fill(text, 78))
print("\n  Problems: free-form shape, no citation, no 'unknown' path, "
      "no way to diff two runs.")

**Step 2 — a role, explicit rules, and an abstention path.** The rule that matters
most is the second one: *say `NOT_STATED` rather than infer.* Without it the model
fills gaps helpfully, which in a compliance context is called fabrication.

In [ ]:
COMPLIANCE_SYSTEM = (
    "You are a compliance analyst for a regulated financial institution.\n\n"
    "Rules:\n"
    "1. Use only facts present in the source. Never infer, estimate, or generalise.\n"
    '2. If a required field is not stated in the source, output exactly "NOT_STATED".\n'
    "3. Quote the source verbatim in every evidence field. Do not paraphrase.\n"
    "4. Regulatory obligations count only if the source names the regulation.\n"
    "5. Write in the past tense, third person. No hedging, no recommendations."
)

COMPLIANCE_SCHEMA = {
    "type": "object",
    "properties": {
        "incident_summary": {"type": "string", "description": "Two sentences maximum."},
        "detected_at_utc": {"type": "string"},
        "mitigated_at_utc": {"type": "string"},
        "regulations_cited": {"type": "array", "items": {"type": "string"}},
        "personal_data_involved": {"type": "boolean"},
        "customer_records_affected": {"type": ["integer", "null"]},
        "notification_deadline_met": {
            "type": "string",
            "enum": ["yes", "no", "NOT_STATED"],
        },
        "root_cause_category": {
            "type": "string",
            "enum": [
                "identity_and_access", "configuration_drift", "software_defect",
                "third_party", "human_error", "NOT_STATED",
            ],
        },
        "evidence": {
            "type": "array",
            "description": "Verbatim quotes supporting each finding.",
            "items": {
                "type": "object",
                "properties": {
                    "field": {"type": "string"},
                    "quote": {"type": "string"},
                },
                "required": ["field", "quote"],
                "additionalProperties": False,
            },
        },
    },
    "required": [
        "incident_summary", "detected_at_utc", "mitigated_at_utc",
        "regulations_cited", "personal_data_involved",
        "customer_records_affected", "notification_deadline_met",
        "root_cause_category", "evidence",
    ],
    "additionalProperties": False,
}

COMPLIANCE_FORMAT = {
    "type": "json_schema",
    "json_schema": {"name": "compliance", "strict": True, "schema": COMPLIANCE_SCHEMA},
}

text, usage, ms = chat(
    f"Produce the compliance record for this incident report.\n\n{INCIDENT}",
    system=COMPLIANCE_SYSTEM,
    response_format=COMPLIANCE_FORMAT,
)
report("constrained", usage, ms)
record = json.loads(text)
print(json.dumps(record, indent=2))

**Step 3 — verify the abstention path actually works.** A rule the model ignores is
not a control. Feed it a report that is missing the mitigation time and confirm you
get `NOT_STATED` rather than a plausible invention.

In [ ]:
THIN = (
    "On 2 April 2026 at 09:00 UTC an alert fired on the reporting service. "
    "An engineer acknowledged it. No customer data was reviewed."
)

text, usage, ms = chat(
    f"Produce the compliance record for this incident report.\n\n{THIN}",
    system=COMPLIANCE_SYSTEM,
    response_format=COMPLIANCE_FORMAT,
)
report("abstention test", usage, ms)
thin = json.loads(text)

for field in ("mitigated_at_utc", "notification_deadline_met", "root_cause_category"):
    value = thin[field]
    ok = "PASS" if value in ("NOT_STATED", None) else "FAIL — invented a value"
    print(f"  {field:28} = {value!r:16} {ok}")
print(f"  regulations_cited            = {thin['regulations_cited']}")

In [ ]:
# Every quote must appear verbatim in the source. This check is cheap and catches
# the single most damaging failure mode.
def verify_evidence(rec, source):
    bad = [e for e in rec["evidence"] if e["quote"] not in source]
    if bad:
        print("UNGROUNDED evidence — reject this record:")
        for e in bad:
            print(f"  {e['field']}: {e['quote'][:70]}")
    else:
        print(f"all {len(rec['evidence'])} quotes verified verbatim against the source")
    return not bad


verify_evidence(record, INCIDENT)
verify_evidence(thin, THIN)

That last cell is the whole responsible-AI argument in six lines: the schema
guarantees *shape*, the system prompt requests *behaviour*, and only the
verification step guarantees *grounding*. Ship all three.

**When to climb further.** If the abstention test fails repeatedly, or your labels
are fixed and you have labelled data, stop prompt-engineering and train a **custom
Language project** (custom NER or custom classification, per the portal walkthrough
in the README). You get stable categories, per-category confidence, and offsets —
and a confusion matrix to argue with. Fine-tuning the model is the *last* step, and
it fixes format and style, not knowledge.

## 11. The decision table, rebuilt from what you just measured

Run this to print the summary you should be able to reproduce from memory on exam
day.

In [ ]:
DECISIONS = [
    ("Exact character offsets for redaction",        "Language recognize_pii_entities"),
    ("Stable, documented entity categories",         "Language recognize_entities"),
    ("Novel or customer-specific categories",        "LLM + Structured Outputs"),
    ("Summary traceable to source sentences",        "Language begin_extract_summary (LRO)"),
    ("Fluent executive summary",                     "Abstractive summarization or LLM"),
    ("Call-centre transcript: issue + resolution",   "Conversational summarization"),
    ("Positive / negative + per-sentence scores",    "Language analyze_sentiment"),
    ("What specifically the customer disliked",      "Opinion mining"),
    ("Emotional register / sarcasm / escalation",    "LLM"),
    ("Hate / self-harm / sexual / violent content",  "Content Safety analyze_text"),
    ("Jailbreak or indirect prompt injection",       "Content Safety Prompt Shields"),
    ("Clinical concepts linked to UMLS",             "begin_analyze_healthcare_entities (LRO)"),
    ("High-volume translation, uniform quality",     "Azure Translator"),
    ("Enforced terminology, a few terms",            "Dynamic dictionary"),
    ("Enforced terminology, domain-wide",            "Custom Translator"),
    ("Register, idiom, marketing voice",             "LLM translation"),
    ("Output must never be malformed JSON",          "Structured Outputs, strict=true"),
    ("Fixed labels + labelled data + confidences",   "Custom Language project"),
]

width = max(len(r) for r, _ in DECISIONS)
print(f"{'Requirement'.ljust(width)}    Answer")
print("-" * (width + 44))
for req, answer in DECISIONS:
    print(f"{req.ljust(width)} -> {answer}")

## What you built

- [x] Entities, key phrases, and linked entities with **offsets**
- [x] The same extraction as **schema-guaranteed** JSON, and the two ways to break a strict schema
- [x] Extractive, abstractive, and LLM summaries — with the LRO pattern
- [x] Sentiment, opinion mining, and LLM tone as three different questions
- [x] PII detection, filtered redaction, and PHI with UMLS links
- [x] Content Safety severities, and why they are not sentiment
- [x] Translator vs LLM measured, plus a dynamic dictionary
- [x] A compliance summarizer with an abstention rule and a grounding check

**Cleanup:** nothing to delete — every service here is consumption-priced and no
resource was created.

**Next:** [04.2 — Implement speech solutions](../02_speech_solutions/README.md),
where the same "Foundry Tool or model?" question returns for audio.

**Check yourself:** [quiz.md](quiz.md)